In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
import requests
import fsspec
from shapely import wkt
import re
from calitp_data_analysis.sql import get_engine
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()


pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/census_tracts_data.parquet", "rb") as f:
    tracts_ca_acs = gpd.read_parquet(f)

In [5]:
# Load the stored organization, ridership, stop, data from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/ridership_trips_routes_weekday.csv", "rb") as f:
    ridership_trips_routes_weekday = pd.read_csv(f)

In [6]:
# Load job density data from GCS and select required columns
# Open the GCS file using your existing fs object
with fs.open(f"{GCS_FILE_PATH}/job_density_2023.parquet", "rb") as f:
    jobdata = pd.read_parquet(f)

# Select only the columns you want, including geometry
jobdata = jobdata[['GEOID', 'jobs_tot']]

## Spatial Analysis: Stop Buffers and Census Tract Intersections

In [7]:
ridership_trips_routes_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21265 entries, 0 to 21264
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         21265 non-null  object 
 1   feed_key                  21264 non-null  object 
 2   stop_id                   21264 non-null  object 
 3   stop_name                 21265 non-null  object 
 4   stop_code                 20543 non-null  object 
 5   n_arrivals                21264 non-null  float64
 6   n_routes                  21264 non-null  float64
 7   pt_geom                   21264 non-null  object 
 8   day_type                  21265 non-null  object 
 9   average_daily_boardings   21265 non-null  float64
 10  average_daily_alightings  19087 non-null  float64
 11  start_date                21265 non-null  object 
 12  end_date                  21265 non-null  object 
dtypes: float64(4), object(9)
memory usage: 2.1+ MB


In [8]:
# Drop rows with missing pt_geom
ridership_trips_routes_weekday = ridership_trips_routes_weekday[
    ridership_trips_routes_weekday['pt_geom'].notna() & 
    (ridership_trips_routes_weekday['pt_geom'] != 'nan')
].copy()


In [9]:
# Ensure pt_geom is string type
ridership_trips_routes_weekday['pt_geom'] = ridership_trips_routes_weekday['pt_geom'].astype(str)

In [10]:
# Convert pt_geom column from WKT to shapely geometry
ridership_trips_routes_weekday['geometry'] = ridership_trips_routes_weekday['pt_geom'].apply(wkt.loads)

# Create a GeoDataFrame
gdf_ridership = gpd.GeoDataFrame(ridership_trips_routes_weekday, geometry='geometry')

In [11]:
# Set CRS (assuming WGS84)
gdf_ridership.set_crs(epsg=4326, inplace=True)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,VNACLR1,.,NaN,14.0,1.0,POINT(-119.294028 34.343645),Weekday,0.000000,3.000000,2025-05-01,2025-05-31,POINT (-119.29403 34.34365)
1,Samtrans,db97cc02836aa5f0cf647d80160b23ec,345017,1000 El Camino Real-Menlo College,345017,72.0,1.0,POINT(-122.191284 37.457543),Weekday,9.523810,24.571429,2025-08-01,2025-08-31,POINT (-122.19128 37.45754)
2,Samtrans,db97cc02836aa5f0cf647d80160b23ec,343119,1011 Crestview Dr,343119,5.0,1.0,POINT(-122.284103 37.484282),Weekday,1.444444,1.888889,2025-08-01,2025-08-31,POINT (-122.28410 37.48428)
3,Samtrans,db97cc02836aa5f0cf647d80160b23ec,340606,1060 Carolan Ave,340606,7.0,1.0,POINT(-122.359627 37.586685),Weekday,10.333333,5.500000,2025-08-01,2025-08-31,POINT (-122.35963 37.58669)
4,SDMTS,1fff52f9349da228c56eef492df5001b,12049,10th Av & A St,12049,18.0,3.0,POINT(-117.15569071 32.71857729),Weekday,3.485637,42.889568,2024-09-01,2025-01-25,POINT (-117.15569 32.71858)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21260,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70241,Santa Clara,NaN,45.0,2.0,POINT(-121.93608 37.353238),Weekday,718.467622,NaN,2023-11-01,2025-07-31,POINT (-121.93608 37.35324)
21261,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70041,South San Francisco,NaN,52.0,3.0,POINT(-122.404979051 37.655941395),Weekday,559.501648,NaN,2023-11-01,2025-07-31,POINT (-122.40498 37.65594)
21262,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70221,Sunnyvale,NaN,52.0,3.0,POINT(-122.031372 37.378916),Weekday,1408.195136,NaN,2023-11-01,2025-07-31,POINT (-122.03137 37.37892)
21263,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70271,Tamien,NaN,23.0,2.0,POINT(-121.883721 37.31174),Weekday,198.883066,NaN,2023-11-01,2025-07-31,POINT (-121.88372 37.31174)


In [12]:
# Reproject to match census tracts CRS
gdf_ridership = gdf_ridership.to_crs(tracts_ca_acs.crs)

In [13]:
stop_buffered = gdf_ridership.copy()
stop_buffered["geometry"] = stop_buffered.geometry.buffer(804.672)

In [14]:
# Inner join with ACS data on 'geo_id'
tracts_ca_acs = tracts_ca_acs.merge(jobdata, on = 'GEOID', how='left')

In [15]:
tracts_ca_acs.crs

<Projected CRS: EPSG:3310>
Name: NAD83 / California Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - California.
- bounds: (-124.45, 32.53, -114.12, 42.01)
Coordinate Operation:
- name: California Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [16]:
stop_buffered.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 21264 entries, 0 to 21264
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   organization_name         21264 non-null  object  
 1   feed_key                  21264 non-null  object  
 2   stop_id                   21264 non-null  object  
 3   stop_name                 21264 non-null  object  
 4   stop_code                 20543 non-null  object  
 5   n_arrivals                21264 non-null  float64 
 6   n_routes                  21264 non-null  float64 
 7   pt_geom                   21264 non-null  object  
 8   day_type                  21264 non-null  object  
 9   average_daily_boardings   21264 non-null  float64 
 10  average_daily_alightings  19087 non-null  float64 
 11  start_date                21264 non-null  object  
 12  end_date                  21264 non-null  object  
 13  geometry                  21264 non-null  g

In [17]:
geometry_intersect = gpd.overlay(
    stop_buffered, 
    tracts_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)

In [18]:
# Calculate intersected area
geometry_intersect['area_2'] = geometry_intersect.geometry.area

# Calculate the proportion of the tract that intersects each stop
geometry_intersect['area_ratio'] = geometry_intersect['area_2'] / geometry_intersect['area_m2']

In [19]:
# Define demographic and socioeconomic columns to be adjusted by area ratio
cols_to_weight = [
    'total_pop', 'poverty_pop', 'non_us_citizen', 'workers_with_no_car', 
    'households_with_no_cars', 'disabled_pop', 'public_asst_pop', 
    'inc_extremelylow', 'inc_verylow', 'inc_low', 
    'male_seniors', 'female_seniors', 'veteran_pop', 'male_youth', 'inc_total_lowincome',  'female_youth', 'total_seniors', 'jobs_tot', 'total_youth', 'ALAND'
]

# Apply area_ratio
for col in cols_to_weight:
    geometry_intersect[f'{col}_adj'] = geometry_intersect[col] * geometry_intersect['area_ratio']

In [20]:
geometry_intersect.organization_name.unique()

array(['Gold Coast Transit', 'Samtrans', 'SDMTS', 'Fresno County',
       'SacRT Bus', 'Orange County Transportation Authority',
       'Long Beach Transit', 'Foothill Transit',
       'Golden Gate Park Shuttle', 'Big Blue Bus', 'Culver City Bus',
       'Riverside Transit', 'Caltrain', 'City of Burbank'], dtype=object)

In [21]:
stop_acs_rollup = geometry_intersect.groupby(
    ['feed_key', 'stop_id', 'organization_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight]].sum()

In [22]:
stop_route_df = gdf_ridership.merge(
    stop_acs_rollup,
    on=['feed_key', 'stop_id','organization_name'],
    how='left'
)

In [23]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 21264 entries, 0 to 21263
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   organization_name            21264 non-null  object  
 1   feed_key                     21264 non-null  object  
 2   stop_id                      21264 non-null  object  
 3   stop_name                    21264 non-null  object  
 4   stop_code                    20543 non-null  object  
 5   n_arrivals                   21264 non-null  float64 
 6   n_routes                     21264 non-null  float64 
 7   pt_geom                      21264 non-null  object  
 8   day_type                     21264 non-null  object  
 9   average_daily_boardings      21264 non-null  float64 
 10  average_daily_alightings     19087 non-null  float64 
 11  start_date                   21264 non-null  object  
 12  end_date                     21264 non-null  object 

In [24]:
stop_route_df.organization_name.unique()

array(['Gold Coast Transit', 'Samtrans', 'SDMTS', 'Fresno County',
       'SacRT Bus', 'Orange County Transportation Authority',
       'Long Beach Transit', 'Foothill Transit',
       'Golden Gate Park Shuttle', 'Big Blue Bus', 'Culver City Bus',
       'Riverside Transit', 'Caltrain', 'City of Burbank'], dtype=object)

In [25]:
stop_route_df = gpd.GeoDataFrame(
    stop_route_df, 
    geometry='geometry', 
    crs=geometry_intersect.crs
)

In [26]:
# Store data in warehouse
with fs.open(f"{GCS_FILE_PATH}/stop_route_df.parquet", "wb") as f:
    stop_route_df.to_parquet(f, index=False)